Install via `pip install pylibcugraphops-cu12`

In [1]:
import dgl
import torch

try:  # pragma: no cover
    LEGACY_MODE = False
    from pylibcugraphops.pytorch import CSC, HeteroCSC

    HAS_PYLIBCUGRAPHOPS = True
except ImportError:
    HAS_PYLIBCUGRAPHOPS = False
    try:  # pragma: no cover
        from pylibcugraphops import make_fg_csr

        LEGACY_MODE = True
    except ImportError:
        pass

In [2]:
assert HAS_PYLIBCUGRAPHOPS and not LEGACY_MODE

In [3]:
import torch.nn as nn

In [4]:
from pylibcugraphops.pytorch import operators

In [5]:
from torch_geometric.edge_index import EdgeIndex

In [6]:
def get_cugraph(
    edge_index: EdgeIndex,
):
    r"""Constructs a :obj:`cugraph` graph object from CSC representation.

    Args:
        edge_index (EdgeIndex): The edge indices.
    """
    if not isinstance(edge_index, EdgeIndex):
        raise ValueError(f"'edge_index' needs to be of type 'EdgeIndex' (got {type(edge_index)})")

    edge_index = edge_index.sort_by("col")[0]
    num_src_nodes = edge_index.get_sparse_size(0)
    (colptr, row), _ = edge_index.get_csc()

    if not row.is_cuda:
        raise RuntimeError("'get_cugraph' requires GPU-based processing (got CPU tensor)")

    if LEGACY_MODE:
        return make_fg_csr(colptr, row)

    return CSC(colptr, row, num_src_nodes=num_src_nodes)

In [7]:
edge_index = EdgeIndex(
    [[0, 0, 1, 1], [1, 2, 0, 2]],
    sparse_size=(3, 3),
    sort_order="row",
    is_undirected=False,
    device="cuda",
)

In [8]:
src = torch.tensor([0, 0, 1, 1])
dst = torch.tensor([1, 2, 0, 2])

features = torch.tensor([[1, 1], [2, 1], [1, 2]]).float().cuda()

In [9]:
graph = dgl.graph((src, dst)).to("cuda")


target = dgl.ops.copy_u_mean(graph, features)

In [10]:
graph_cugraph = get_cugraph(edge_index=edge_index)
graph_dgl = graph

In [11]:
dir(operators)

['FusedTensorProductOp3',
 'FusedTensorProductOp4',
 'SymmetricTensorContraction',
 '__all__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 'agg_concat',
 'agg_concat_e2n',
 'agg_concat_n2n',
 'agg_concat_n2n_e2n',
 'agg_dmpnn',
 'agg_dmpnn_e2e',
 'agg_hg_basis',
 'agg_hg_basis_n2n_post',
 'agg_simple',
 'agg_simple_e2n',
 'agg_simple_n2n',
 'agg_simple_n2n_e2n',
 'fused_tensor_product',
 'fused_tensor_product_transpose',
 'mha_gat',
 'mha_gat_n2n',
 'mha_gat_v2',
 'mha_gat_v2_n2n',
 'mha_simple',
 'mha_simple_n2n',
 'symmetric_tensor_contraction',
 'transpose_segment_to_contiguous',
 'transpose_segment_to_strided',
 'update_efeat',
 'update_efeat_e2e']

In [12]:
target = dgl.ops.copy_u_mean(graph, features)
target

tensor([[2.0000, 1.0000],
        [1.0000, 1.0000],
        [1.5000, 1.0000]], device='cuda:0')

In [13]:
operators.agg_simple_n2n(features, graph_cugraph, "mean")

tensor([[2.0000, 1.0000],
        [1.0000, 1.0000],
        [1.5000, 1.0000]], device='cuda:0')

In [14]:
?operators.mha_gat_v2_n2n

Signature:
operators.mha_gat_v2_n2n(
    feat: Union[torch.Tensor, Tuple[torch.Tensor]],
    attn_weights: torch.Tensor,
    graph: pylibcugraphops.pytorch.graph.CSC,
    num_heads: int = 1,
    activation: str = 'LeakyReLU',
    negative_slope: float = 0.2,
    concat_heads: bool = True,
    edge_feat: Optional[torch.Tensor] = None,
    deterministic_dgrad: Optional[bool] = False,
    deterministic_wgrad: Optional[bool] = False,
) -> torch.Tensor
Docstring:
PyTorch autograd function for a multi-head attention layer (GAT-like)
without using cudnn (mha_gat_v2) with an activation prior to the dot
product but none afterwards in a node-to-node reduction (n2n).

Parameters
----------
feat : torch.Tensor | Tuple[torch.Tensor, torch.Tensor]
    The input node features.
    Depending on whether the graph is bipartite or not, this has
    to be a tuple of two tensors or a single tensor correspondingly.
    For bipartite graphs, the rows of each tensors consists of
    concatenated features from

In [15]:
?dgl.nn.GATv2Conv

Init signature:
dgl.nn.GATv2Conv(
    in_feats,
    out_feats,
    num_heads,
    feat_drop=0.0,
    attn_drop=0.0,
    negative_slope=0.2,
    residual=False,
    activation=None,
    allow_zero_in_degree=False,
    bias=True,
    share_weights=False,
)
Docstring:     
GATv2 from `How Attentive are Graph Attention Networks?
<https://arxiv.org/pdf/2105.14491.pdf>`__

.. math::
    h_i^{(l+1)} = \sum_{j\in \mathcal{N}(i)} \alpha_{ij}^{(l)} W^{(l)}_{right} h_j^{(l)}

where :math:`\alpha_{ij}` is the attention score bewteen node :math:`i` and
node :math:`j`:

.. math::
    \alpha_{ij}^{(l)} &= \mathrm{softmax_i} (e_{ij}^{(l)})

    e_{ij}^{(l)} &= {\vec{a}^T}^{(l)}\mathrm{LeakyReLU}\left(
        W^{(l)}_{left} h_{i} + W^{(l)}_{right} h_{j}\right)

Parameters
----------
in_feats : int, or pair of ints
    Input feature size; i.e, the number of dimensions of :math:`h_i^{(l)}`.
    If the layer is to be applied to a unidirectional bipartite graph, `in_feats`
    specifies the input feature 

In [16]:
features_for_gat = torch.cat([features.clone(), features.clone() + 1], 1)  # 2 heads
features_for_gat

tensor([[1., 1., 2., 2.],
        [2., 1., 3., 2.],
        [1., 2., 2., 3.]], device='cuda:0')

In [17]:
attention_weights = nn.Parameter(torch.tensor([1, 1, 1, 1]).float()).cuda()
attention_weights

tensor([1., 1., 1., 1.], device='cuda:0', grad_fn=<ToCopyBackward0>)

In [18]:
operators.mha_gat_v2_n2n(features_for_gat, attention_weights, graph_cugraph, 2)

tensor([[2.0000, 1.0000, 3.0000, 2.0000],
        [1.0000, 1.0000, 2.0000, 2.0000],
        [1.7311, 1.0000, 2.7311, 2.0000]], device='cuda:0',
       grad_fn=<_mha_gat_v2_n2n_autogradBackward>)

In [19]:
?operators.mha_simple_n2n

Signature:
operators.mha_simple_n2n(
    key_emb: torch.Tensor,
    query_emb: torch.Tensor,
    value_emb: torch.Tensor,
    graph: pylibcugraphops.pytorch.graph.CSC,
    num_heads: int = 1,
    concat_heads: bool = True,
    edge_emb: Optional[torch.Tensor] = None,
    norm_by_dim: bool = True,
    score_bias: Optional[torch.Tensor] = None,
    score_weight: Optional[torch.Tensor] = None,
) -> torch.Tensor
Docstring:
PyTorch autograd function for a multi-head attention layer (mha_simple)
without using cudnn with an activation prior to the dot
product but none afterwards in a node-to-node reduction (n2n).

Parameters
----------
key_emb : torch.Tensor
    The key embeddings of input nodes. Each row consists of concatenated features
    from different heads after the linear transformation.
    Shape: (n_src_nodes, dim_in) where dim_in = dim_head * num_heads, with
            dim_head being the feature dimension per head.

query_emb : torch.Tensor
    The query embeddings of input nodes.

In [ ]:
?operators.agg_simple_n2n

In [21]:
?operators.mha_simple_n2n

Signature:
operators.mha_simple_n2n(
    key_emb: torch.Tensor,
    query_emb: torch.Tensor,
    value_emb: torch.Tensor,
    graph: pylibcugraphops.pytorch.graph.CSC,
    num_heads: int = 1,
    concat_heads: bool = True,
    edge_emb: Optional[torch.Tensor] = None,
    norm_by_dim: bool = True,
    score_bias: Optional[torch.Tensor] = None,
    score_weight: Optional[torch.Tensor] = None,
) -> torch.Tensor
Docstring:
PyTorch autograd function for a multi-head attention layer (mha_simple)
without using cudnn with an activation prior to the dot
product but none afterwards in a node-to-node reduction (n2n).

Parameters
----------
key_emb : torch.Tensor
    The key embeddings of input nodes. Each row consists of concatenated features
    from different heads after the linear transformation.
    Shape: (n_src_nodes, dim_in) where dim_in = dim_head * num_heads, with
            dim_head being the feature dimension per head.

query_emb : torch.Tensor
    The query embeddings of input nodes.